# Probe Analysis

Train a logistic-regression probe on JudgementLM attention activations and evaluate
its accuracy at predicting valid/invalid labels. Includes calibration and greedy head selection.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET         = 'pond'
EXTRACTION_MODEL = 'gemma-3-27b'
EXTRACTION_DATE  = '2026_04_01'
JUDGE_MODEL      = 'llama-3.1-8b'

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from analysis.loaders import load_activations, load_combined_judgements
from analysis.plots import probe_accuracy_heatmap, calibration_curve, probability_distribution
from scholarlm.utils.probe import build_feature_matrix, train_probe, eval_probe
from scholarlm.utils.calibration import reliability_diagram_data

## Load activations and labels

In [ ]:
activations = load_activations(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE, JUDGE_MODEL)
judgements  = load_combined_judgements(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE)
judged_df   = pd.DataFrame(judgements)

# Align: keep only records that have activation entries
records_with_act = judged_df[judged_df['measurement_id'].astype(str).isin(activations.files)]
measurement_ids  = list(records_with_act['measurement_id'])
labels           = records_with_act['judgement_combined'].values.astype(bool)

print(f'n={len(measurement_ids)}  positive rate={labels.mean():.2%}')

In [ ]:
X = build_feature_matrix(activations, measurement_ids)
print(f'Feature matrix: {X.shape}')

## Train and evaluate (k-fold)

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accs = []
for train_idx, test_idx in kf.split(X, labels):
    probe = train_probe(X[train_idx], labels[train_idx])
    result = eval_probe(probe, X[test_idx], labels[test_idx])
    fold_accs.append(result['accuracy'])

print(f'5-fold accuracy: {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}')

## Calibration

In [ ]:
# Train on 80%, calibrate on 20%
split = int(0.8 * len(X))
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
probe = train_probe(X[idx[:split]], labels[idx[:split]])

# Get predicted probabilities on held-out set
probs = probe.predict_proba(X[idx[split:]])[:, 1]
diag  = reliability_diagram_data(probs, labels[idx[split:]])
print(f'ECE = {diag["ece"]:.4f}')

fig = calibration_curve(diag, judge_labels=[JUDGE_MODEL])
Path('figures').mkdir(exist_ok=True)
fig.savefig('figures/calibration.pdf', bbox_inches='tight')
fig.show()

## Greedy head selection

Iteratively add the single attention head that most improves probe accuracy.

In [ ]:
# Infer activation shape from the first entry
sample_key = next(iter(activations.files))
arr = activations[sample_key]         # (n_layers, n_heads, head_dim)
n_layers, n_heads, head_dim = arr.shape

def probe_single_head(layer, head):
    """Build feature matrix using only one (layer, head) pair."""
    rows = []
    for mid in measurement_ids:
        a = activations[str(mid)]  # (n_layers, n_heads, head_dim)
        rows.append(a[layer, head, :])
    X_head = np.stack(rows).astype(np.float32)
    accs = []
    for tr, te in kf.split(X_head, labels):
        p = train_probe(X_head[tr], labels[tr])
        accs.append(eval_probe(p, X_head[te], labels[te])['accuracy'])
    return np.mean(accs)

# Score every head (may take a minute)
head_scores = {}
for l in range(n_layers):
    for h in range(n_heads):
        head_scores[(l, h)] = probe_single_head(l, h)

top_heads = sorted(head_scores, key=head_scores.get, reverse=True)[:10]
print('Top 10 heads by single-head probe accuracy:')
for lh in top_heads:
    print(f'  layer={lh[0]:2d} head={lh[1]:2d}  acc={head_scores[lh]:.4f}')

In [ ]:
# Visualise all head accuracies as a heatmap
import matplotlib.pyplot as plt

heat = np.full((n_layers, n_heads), np.nan)
for (l, h), acc in head_scores.items():
    heat[l, h] = acc

fig, ax = plt.subplots(figsize=(max(6, n_heads * 0.4), max(4, n_layers * 0.4)))
im = ax.imshow(heat, vmin=0.5, vmax=1.0, cmap='viridis', aspect='auto')
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title(f'Single-head probe accuracy ({JUDGE_MODEL})')
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig('figures/head_accuracy_heatmap.pdf', bbox_inches='tight')
fig.show()